[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/InteligenciaArtificial/blob/main/tutoriales/notebooks/apis-avanzadas/05-prompt-caching.ipynb)

# Prompt Caching: Hasta 90% de Reducción en Costes

> Aprende a cachear partes estáticas del contexto para reducir costes
> y latencia en aplicaciones con system prompts o documentos grandes.

## Objetivos
- Marcar bloques de contexto para caché con `cache_control`
- Medir el ahorro real en tokens y costes
- Implementar caché en conversaciones multi-turn
- Calcular el umbral de rentabilidad del caching

## 1. Instalación de dependencias

In [ ]:
%pip install anthropic --quiet

## 2. Primera llamada: escribir en caché

In [ ]:
import anthropic
import time

client = anthropic.Anthropic()

# Contexto largo que queremos cachear (manual de empresa)
CONTEXTO_LARGO = """
MANUAL CORPORATIVO — TechCorp SA — Versión 8.1 (2025)

MISIÓN Y VALORES
TechCorp es una empresa de software B2B fundada en 2015. Nuestra misión es democratizar
el acceso a tecnología avanzada para pymes europeas. Nuestros valores son: innovación
continua, transparencia con clientes, bienestar del equipo y sostenibilidad digital.

PRODUCTOS Y SERVICIOS
- CloudManager Pro: plataforma de gestión cloud multi-proveedor (AWS, Azure, GCP)
  Precio: 299-2.999€/mes según instancias gestionadas
- DataPipeline: ETL automatizado con conectores para 200+ fuentes de datos
  Precio: 499€/mes + 0.01€/GB procesado
- AIAssistant Enterprise: suite de IA para automatización de procesos internos
  Precio: 999€/mes por equipo de hasta 20 usuarios

POLÍTICA COMERCIAL
- Descuentos por volumen: 10% (>5 licencias), 20% (>20 licencias), 30% (>100 licencias)
- Período de prueba gratuita: 30 días sin tarjeta de crédito
- Contratos anuales con descuento adicional del 15%
- SLA garantizado: 99.9% uptime, soporte 24/7 para Enterprise

PROCESO DE VENTAS
1. Lead qualification (1-2 días)
2. Demo personalizada (30-60 min)
3. Propuesta técnica y económica (3-5 días)
4. Negociación y cierre (1-4 semanas según tamaño)
5. Onboarding técnico (5-15 días según producto)

COMPETIDORES PRINCIPALES
- CloudBase: similar a CloudManager pero sin multi-cloud. Precio 30% más bajo.
- DataFlow Pro: competidor directo de DataPipeline. Mejor UI, menos conectores.
- BotifAI: competidor de AIAssistant, fuerte en sector retail.

DIFERENCIADORES CLAVE
- Único en ofrecer los 3 productos integrados con single sign-on
- Soporte en español, catalán, francés y portugués
- Servidores en Europa (GDPR compliance nativo)
- API pública y webhooks para integraciones personalizadas
"""

def llamada_con_cache(pregunta: str, mostrar_metricas: bool = True) -> dict:
    """Llama a la API con prompt caching habilitado."""
    inicio = time.perf_counter()
    resp = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=300,
        system=[
            {
                "type": "text",
                "text": "Eres un asistente comercial experto en los productos de TechCorp. "
                        "Responde de forma concisa y precisa usando solo la información del manual."
            },
            {
                "type": "text",
                "text": CONTEXTO_LARGO,
                "cache_control": {"type": "ephemeral"}   # ← Activar caché aquí
            }
        ],
        messages=[{"role": "user", "content": pregunta}]
    )
    latencia = time.perf_counter() - inicio

    cache_write = getattr(resp.usage, "cache_creation_input_tokens", 0)
    cache_read = getattr(resp.usage, "cache_read_input_tokens", 0)

    if mostrar_metricas:
        print(f"  cache_write: {cache_write:,} | cache_read: {cache_read:,} | "
              f"input: {resp.usage.input_tokens:,} | output: {resp.usage.output_tokens:,} | "
              f"latencia: {latencia:.2f}s")

    return {
        "respuesta": resp.content[0].text,
        "cache_write": cache_write,
        "cache_read": cache_read,
        "input_tokens": resp.usage.input_tokens,
        "output_tokens": resp.usage.output_tokens,
        "latencia": latencia
    }

print("=== PRUEBA DE PROMPT CACHING ===")
print("\nLlamada 1 (escritura en caché):")
r1 = llamada_con_cache("¿Cuánto cuesta CloudManager Pro?")
print(f"  → {r1['respuesta'][:100]}")

print("\nLlamada 2 (lectura desde caché — debería mostrar cache_read > 0):")
r2 = llamada_con_cache("¿Qué descuento hay para contratos anuales?")
print(f"  → {r2['respuesta'][:100]}")

print("\nLlamada 3 (caché en caliente):")
r3 = llamada_con_cache("¿En qué se diferencia TechCorp de CloudBase?")
print(f"  → {r3['respuesta'][:100]}")

## 3. Comparar costes: con y sin caché

In [ ]:
def calcular_coste_haiku(cache_write, cache_read, input_normal, output):
    """Calcula coste en USD para Claude Haiku (precios 2025 por 1M tokens)."""
    return (
        cache_write  * 0.30  +   # Cache write: +25% sobre normal
        cache_read   * 0.03  +   # Cache read: -88% respecto normal
        input_normal * 0.25  +   # Input normal
        output       * 1.25      # Output (sin descuento)
    ) / 1_000_000

def calcular_coste_sin_cache(input_tokens, output_tokens):
    """Coste sin caché (precio normal)."""
    return (input_tokens * 0.25 + output_tokens * 1.25) / 1_000_000

resultados = [r1, r2, r3]
tokens_contexto = len(CONTEXTO_LARGO.split()) * 1.3  # estimación tokens

print("=== COMPARATIVA DE COSTES ===")
print(f"{'Llamada':<10} {'Sin caché ($)':<15} {'Con caché ($)':<15} {'Ahorro (%)':<12}")
print("-" * 55)

total_sin = 0
total_con = 0
for i, r in enumerate(resultados, 1):
    tokens_input_total = r['cache_write'] + r['cache_read'] + r['input_tokens']
    c_sin = calcular_coste_sin_cache(tokens_input_total, r['output_tokens'])
    c_con = calcular_coste_haiku(r['cache_write'], r['cache_read'],
                                  r['input_tokens'], r['output_tokens'])
    ahorro = (1 - c_con / c_sin) * 100 if c_sin > 0 else 0
    print(f"Llamada {i:<4} ${c_sin:<14.6f} ${c_con:<14.6f} {ahorro:<12.1f}%")
    total_sin += c_sin
    total_con += c_con

print("-" * 55)
ahorro_total = (1 - total_con / total_sin) * 100 if total_sin > 0 else 0
print(f"{'TOTAL':<10} ${total_sin:<14.6f} ${total_con:<14.6f} {ahorro_total:<12.1f}%")

print(f"\nProyección a 1.000 llamadas:")
print(f"  Sin caché: ${total_sin/3*1000:.2f}")
print(f"  Con caché: ${total_con/3*1000:.2f}")
print(f"  Ahorro: ${(total_sin-total_con)/3*1000:.2f}")

## 4. Caché en conversación multi-turn

In [ ]:
# Conversación donde cacheamos el historial previo
historial = []

def chat_con_cache_progresivo(mensaje: str) -> str:
    """Chat que cachea el system prompt y los últimos turns del asistente."""
    historial.append({"role": "user", "content": mensaje})

    # Construir mensajes con cache_control en el turn anterior del asistente
    mensajes_api = []
    for i, msg in enumerate(historial):
        if msg["role"] == "assistant" and i == len(historial) - 2:
            # Cachear el último turno del asistente
            mensajes_api.append({
                "role": "assistant",
                "content": [{
                    "type": "text",
                    "text": msg["content"],
                    "cache_control": {"type": "ephemeral"}
                }]
            })
        else:
            mensajes_api.append(msg)

    resp = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=300,
        system=[
            {"type": "text", "text": "Eres un asistente comercial de TechCorp.",
             "cache_control": {"type": "ephemeral"}},
            {"type": "text", "text": CONTEXTO_LARGO,
             "cache_control": {"type": "ephemeral"}}
        ],
        messages=mensajes_api
    )

    respuesta = resp.content[0].text
    historial.append({"role": "assistant", "content": respuesta})

    cache_read = getattr(resp.usage, "cache_read_input_tokens", 0)
    print(f"  [cache_read: {cache_read:,} tokens]")
    return respuesta

print("=== CONVERSACIÓN MULTI-TURN CON CACHÉ ===")
turnos = [
    "Hola, estoy evaluando vuestros productos. ¿Qué me recomendáis para gestión cloud?",
    "¿Y cuál sería el coste aproximado para nuestra empresa de 50 empleados?",
    "¿Podemos probar el producto antes de comprar?",
]

for turno in turnos:
    print(f"\nUsuario: {turno}")
    respuesta = chat_con_cache_progresivo(turno)
    print(f"Asistente: {respuesta[:150]}...")

## 5. Calcular el umbral de rentabilidad

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Umbral: ¿a partir de cuántas llamadas vale la pena cachear?
# La primera llamada con cache_write cuesta más (+25%)
# Todas las siguientes ahorran ~88% en tokens cacheados

tokens_sistema = 800    # tokens del system prompt/contexto
tokens_pregunta = 50    # tokens variables por llamada
tokens_respuesta = 100  # tokens de output

# Precio Haiku por 1M tokens
p = {"normal": 0.25, "cache_write": 0.30, "cache_read": 0.03, "output": 1.25}

n_llamadas = np.arange(1, 51)

# Sin caché: pagar tokens_sistema + tokens_pregunta en cada llamada
coste_sin = n_llamadas * (tokens_sistema + tokens_pregunta) * p["normal"] / 1e6 + \
            n_llamadas * tokens_respuesta * p["output"] / 1e6

# Con caché: 1 cache_write + (n-1) cache_reads + tokens_pregunta siempre normal
coste_con = (tokens_sistema * p["cache_write"] / 1e6 +  # 1ª escritura
             (n_llamadas - 1) * tokens_sistema * p["cache_read"] / 1e6 +  # lecturas
             n_llamadas * tokens_pregunta * p["normal"] / 1e6 +  # pregunta variable
             n_llamadas * tokens_respuesta * p["output"] / 1e6)  # output

# Punto de cruce
cruces = np.where(np.diff(np.sign(coste_sin - coste_con)))[0]
punto_equilibrio = cruces[0] + 1 if len(cruces) > 0 else None

plt.figure(figsize=(10, 5))
plt.plot(n_llamadas, coste_sin * 1000, "r-o", markersize=4, label="Sin caché")
plt.plot(n_llamadas, coste_con * 1000, "g-o", markersize=4, label="Con caché")
if punto_equilibrio:
    plt.axvline(x=punto_equilibrio, color="orange", linestyle="--",
                label=f"Punto equilibrio: llamada {punto_equilibrio}")
plt.xlabel("Número de llamadas")
plt.ylabel("Coste acumulado (m$)")
plt.title(f"Coste acumulado: {tokens_sistema} tokens sistema, {tokens_pregunta} tokens pregunta")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("prompt_caching_roi.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nPunto de equilibrio: a partir de la llamada {punto_equilibrio}, el caché empieza a ser rentable")
print(f"Ahorro a 50 llamadas: ${(coste_sin[-1] - coste_con[-1])*1000:.3f}m$ ({(1-coste_con[-1]/coste_sin[-1])*100:.1f}%)")
print("\nRegla práctica: el caché vale la pena cuando tienes >1.024 tokens estáticos")
print("y haces más de 2-3 llamadas con el mismo contexto en una ventana de 5 minutos.")